In [ ]:
import sys
!{sys.executable} -m pip install pandas selenium webdriver-manager openpyxl

In [ ]:
import pandas as pd
import random
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import UnexpectedAlertPresentException, WebDriverException
import glob
import re
import time
import datetime

In [ ]:
def criar_driver():
    service = Service("/usr/bin/chromedriver") #local onde estiver o driver baixado na máquina
    options = webdriver.ChromeOptions()
    options.binary_location = "/usr/bin/chromium-browser" #local onde estiver o binário do driver na máquina
    options.add_argument("--start-maximized")
    # options.add_argument("--headless") # rodar headed reduz ação de bloqueio no PJe por anti-bot
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-blink-features=AutomationControlled")
    return webdriver.Chrome(service=service, options=options)

In [ ]:
# arquivo com numero, is_processed, updated_at para controle do que foi processado e o que deve ser processado na próxima execução
df_origem = pd.read_csv("/registros_processados.csv", dtype=str) 
processos = df_origem['numero'] #numero do processo

In [ ]:
COLUNAS = ["Número do Processo", "Data de Distribuição", "Classe Judicial", "Assunto", 
           "Jurisdição", "Órgão Julgador", "Status", "Polo Ativo", "Polo Passivo", "Outros Interessados"]

In [ ]:
def extrair_participantes(driver, xpath_table):
    participantes = []
    rows = driver.find_elements(By.XPATH, xpath_table)
    for row in rows:
        try:
            name = row.find_element(By.XPATH, ".//td[1]//span[contains(@class,'text-bold') or contains(@class,'')]").text.strip()
            status_p = row.find_element(By.XPATH, ".//td[2]//div").text.strip()
            participantes.append(f"{name} [{status_p}]")
        except:
            continue
    return ", ".join(participantes) if participantes else ""

In [ ]:
def buscar_processo(driver, wait, processo_num):
    input_pesquisa = wait.until(EC.visibility_of_element_located(
        (By.ID, "fPP:numProcesso-inputNumeroProcessoDecoration:numProcesso-inputNumeroProcesso")
    ))

    driver.execute_script("arguments[0].value = '';", input_pesquisa)
    input_pesquisa.click()
    time.sleep(0.5)

    for digito in processo_num:
        input_pesquisa.send_keys(digito)

    time.sleep(1)

    btn_pesquisa = wait.until(EC.element_to_be_clickable((By.ID, "fPP:searchProcessos")))
    btn_pesquisa.click()
    time.sleep(2)

In [ ]:
def abrir_detalhes(driver, wait):
    try:
        link = wait.until(EC.element_to_be_clickable(
            (By.XPATH, "//a[@title='Ver Detalhes']")
        ))
        driver.execute_script("arguments[0].click();", link)
        return True
    except:
        return False

In [ ]:
def trocar_para_nova_aba(driver, wait):
    original = driver.current_window_handle
    wait.until(EC.number_of_windows_to_be(2))

    for handle in driver.window_handles:
        if handle != original:
            driver.switch_to.window(handle)
            return original

In [ ]:
def extrair_dados_processo(driver):
    res = {}

    res["Data de Distribuição"] = driver.find_element(
        By.XPATH, "//label[contains(text(), 'Data da Distribuição')]/../../div[2]"
    ).text.strip()

    res["Classe Judicial"] = driver.find_element(
        By.XPATH, "//label[contains(text(), 'Classe Judicial')]/../../div[2]"
    ).text.strip()

    res["Assunto"] = driver.find_element(
        By.XPATH, "//label[contains(text(), 'Assunto')]/../../div[2]/div"
    ).text.strip()

    res["Jurisdição"] = driver.find_element(
        By.XPATH, "//label[contains(text(), 'Jurisdição')]/../../div[2]"
    ).text.strip()

    res["Órgão Julgador"] = driver.find_element(
        By.XPATH, "//div[contains(@class, 'value') and .//b[text()='Órgão Julgador']]/div"
    ).text.replace("Órgão Julgador", "").strip()

    tb_status = driver.find_elements(
        By.XPATH, "//tbody[contains(@id, 'processoEvento')]/tr/td//span"
    )

    res["Status"] = tb_status[0].text.strip() if tb_status else "Sem movimentação"

    res["Polo Ativo"] = extrair_participantes(driver, "//tbody[contains(@id, 'processoPartesPoloAtivoResumidoTableBinding')]/tr")
    res["Polo Passivo"] = extrair_participantes(driver, "//tbody[contains(@id, 'processoPartesPoloPassivoResumidoTableBinding')]/tr")
    res["Outros Interessados"] = extrair_participantes(driver, "//tbody[contains(@id, 'processoParteOutrosInteressadosResumidoTableBinding')]/tr")

    return res

In [ ]:
def verificar_nao_encontrado(driver):
    msg = driver.find_element(By.XPATH, "//span[contains(text(), 'nenhum processo disponível')]")
    if msg:
        return True
    return False

In [ ]:
def processar_processo(driver, wait, processo_num):
    res = {col: "" for col in COLUNAS}
    res["Número do Processo"] = processo_num

    try:
        buscar_processo(driver, wait, processo_num)

        if not abrir_detalhes(driver, wait):
            if verificar_nao_encontrado(driver):
                res["Status"] = "Não encontrado"
            else:
                res["Status"] = "Erro no carregamento"
            return res

        original = trocar_para_nova_aba(driver, wait)

        dados = extrair_dados_processo(driver)
        res.update(dados)

        driver.close()
        driver.switch_to.window(original)

    except Exception as e:
        res["Status"] = f"Erro: {type(e).__name__}"

        if len(driver.window_handles) > 1:
            try:
                driver.close()
                driver.switch_to.window(driver.window_handles[0])
            except:
                pass

    return res

In [ ]:
def processar_lote_registros():
    """
        Método para processar lote de registros que ainda não foram raspados. Quando o PJe bloquear a automação (o que não demora muito), 
        fica registrado um novo ponto de partida para o script retomar execução
    """
    processos_pendentes = df_origem[df_origem['is_processed'] == 'False']
    
    if processos_pendentes.empty:
        print("🎉 Não há registros pendentes!")
        return

    resultados = []
    driver = criar_driver()
    wait = WebDriverWait(driver, 20)
    
    print(f"🚀 Iniciando processamento. Restam {len(processos_pendentes)} processos.")

    try:
        driver.get("https://pje1g.trf1.jus.br/consultapublica/ConsultaPublica/listView.seam")
        
        for index, row in processos_pendentes.iterrows():
            processo_num = row['numero']
            print(f"🔍 Processando {processo_num} (Índice: {index})...")

            try:
                res = processar_processo(driver, wait, processo_num)
                
                if "Erro:" in res.get("Status", ""):
                    print(f"⚠️ Falha na extração de {processo_num}: {res['Status']}. Tentará novamente na próxima execução.")
                    if "invalid session id" in res['Status'].lower():
                        raise WebDriverException("Sessão expirada ou inválida.")
                    continue 

                resultados.append(res)
                df_origem.at[index, 'is_processed'] = 'True'
                print(f"✅ Sucesso: {processo_num} -> {res.get('Status', '')}")
                
                time.sleep(random.uniform(4, 8))

            except UnexpectedAlertPresentException:
                print(f"🚨 Alerta detectado no processo {processo_num}. Tentando seguir assim mesmo...")
                try:
                    driver.switch_to.alert.accept()
                except:
                    pass
                continue

            except Exception as e:
                print(f"❌ Erro crítico/Sessão encerrada no processo {processo_num}: {e}")
                break

    finally:
        if resultados:
            timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
            nome_saida = f"extracao_{timestamp}.xlsx"
            pd.DataFrame(resultados).to_excel(nome_saida, index=False)
            df_origem.to_csv("/registros_processados.csv", index=False)
            print(f"\n💾 {len(resultados)} registros salvos em {nome_saida}")
        
        try:
            driver.quit()
        except:
            pass

In [ ]:
processar_lote_registros()

In [ ]:
arquivos = glob.glob("/extracao_*.xlsx")

df_final = pd.concat(
    [pd.read_excel(f, dtype=str) for f in arquivos],
    ignore_index=True
)

df_final = df_final.drop_duplicates(subset=["Número do Processo"])

df_final.to_excel(
    "/trf1_completos.xlsx",
    index=False
)

print("Arquivos unificados!")